In [2]:
import pandas as pd
import numpy as np

In [3]:
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

In [4]:
orders.shape, order_items.shape, products.shape, customers.shape, sellers.shape, payments.shape, reviews.shape

((99441, 8),
 (112650, 7),
 (32951, 9),
 (99441, 5),
 (3095, 4),
 (103886, 5),
 (99224, 7))

In [5]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [6]:
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

### Converting string to datetime format: 

In [7]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

In [8]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [9]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [10]:
orders["order_status"].unique()

<StringArray>
[  'delivered',    'invoiced',     'shipped',  'processing', 'unavailable',
    'canceled',     'created',    'approved']
Length: 8, dtype: str

In [11]:
customers["customer_state"].value_counts()

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

In [12]:
customers["customer_city"].nunique()

4119

In [13]:
customers["customer_city"].head(20)

0                    franca
1     sao bernardo do campo
2                 sao paulo
3           mogi das cruzes
4                  campinas
5            jaragua do sul
6                 sao paulo
7                   timoteo
8                  curitiba
9            belo horizonte
10            montes claros
11           rio de janeiro
12         lencois paulista
13                sao paulo
14            caxias do sul
15               piracicaba
16           rio de janeiro
17                guarulhos
18                sao paulo
19                   pacaja
Name: customer_city, dtype: str

In [14]:
customers["customer_city"].sample(20, random_state=42)

52263          sao paulo
46645               unai
37546              icara
94756    duque de caxias
14771              mutum
36263       campo grande
98556            jundiai
23747            diadema
50315          sao paulo
6501           castanhal
11587          sao paulo
65741     rio de janeiro
82092               maua
78549          sao paulo
13695           valinhos
74482           garopaba
51244       novo oriente
36938             recife
22461            goiania
74621     belo horizonte
Name: customer_city, dtype: str

In [15]:
print("Missing values:", customers["customer_city"].isna().sum())

Missing values: 0


In [16]:
print(
    "Leading/trailing whitespace:",
    customers["customer_city"].str.strip().ne(
        customers["customer_city"]
    ).sum()
)

Leading/trailing whitespace: 0


In [17]:
print(
    "Non-lowercase values:",
    (~customers["customer_city"].eq(
        customers["customer_city"].str.lower()
    )).sum()
)

Non-lowercase values: 0


In [18]:
print(
    "Multiple spaces:",
    customers["customer_city"].str.contains(
        r"\s{2,}", regex=True
    ).sum()
)

Multiple spaces: 0


In [19]:
city_state_check = (
    customers.groupby("customer_city")["customer_state"]
    .nunique()
    .sort_values(ascending=False)
)

city_state_check.head(20)

customer_city
sao francisco          4
sao domingos           4
santa maria            4
vera cruz              4
bonito                 4
planalto               4
boa esperanca          4
vicosa                 3
agua branca            3
mundo novo             3
cantagalo              3
sao joao do paraiso    3
sobradinho             3
praia grande           3
santa luzia            3
bom jardim             3
belem                  3
bom jesus              3
triunfo                3
itambe                 3
Name: customer_state, dtype: int64

### Checking the missing data in the tables

In [20]:
print("ORDER ITEMS")
print(order_items.isna().sum())

print("\nPRODUCTS")
print(products.isna().sum())

print("\nPAYMENTS")
print(payments.isna().sum())

print("\nREVIEWS")
print(reviews.isna().sum())

ORDER ITEMS
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

PRODUCTS
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

PAYMENTS
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

REVIEWS
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64


Give me only the rows where product_category_name is missing.

products[
    products["product_category_name"].isna()
]

In [21]:
products[
    products["product_category_name"].isna()
].isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64

### All 610 products missing a category also have their name length, description length, and photo count missing.

### So our next step is to inspect the actual 610 records and, separately, the 2 physical-metadata records.

In [22]:
products[
    products["product_weight_g"].isna()
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
missing_phy_meta_products = products[
    products["product_weight_g"].isna()
]["product_id"]

In [24]:
order_items[
    order_items["product_id"].isin(missing_phy_meta_products)
]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
7098,101157d4fae1c9fb74a00a5dee265c25,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-04-11 08:02:26,29.0,14.52
9233,1521c6bb7b1028154c8c67cf80fa809f,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-04-07 10:10:16,29.0,16.05
28715,415cfaaaa8cea49f934470548797fed1,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-04-07 10:35:19,29.0,14.52
28716,415cfaaaa8cea49f934470548797fed1,2,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-04-07 10:35:19,29.0,14.52
39299,595316a07cd3dea9db7adfcc7e247ae7,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-08-18 04:26:04,39.0,9.27
48424,6e150190fbe04c642a9cf0b80d83ee16,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-06-30 16:45:14,39.0,16.79
48980,6f497c40431d5fb0cfbd6c943dd29215,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-04-11 05:55:32,29.0,10.96
58833,85f8ad45e067abd694b627859fa57453,1,09ff539a621711667c43eba6a3bd8466,8b8cfc8305aa441e4239358c9f6f2485,2017-02-03 21:40:02,1934.0,27.00
71134,a2456e7f02197951664897a94c87242d,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-04-06 11:50:09,29.0,24.84
73556,a7a43f469c0d7bdb0a23a82db125aefa,1,5eb564652db742ff8f28759cd8d2652a,4e922959ae960d389249c378d1c939f5,2017-08-28 13:15:11,39.0,15.10


### Both products with missing physical metadata were actually sold.
### So this isn't some unused catalog record that we can conveniently ignore. It's a real product involved in transactions.
## we retain the missing values.

### Now check whether those 610 products are actually being sold too.

In [25]:
missing_cate_products = products[
    products["product_category_name"].isna()
]["product_id"]

order_items[
    order_items["product_id"].isin(missing_cate_products)
].shape



(1603, 7)

### 1,603 transaction rows involve products whose category is missing.
If we simply drop them, we'd be throwing away valid sales.
If we assign them to some random category, we'd be inventing data.
### So the correct approach is:

Keep them, and represent their category as "Unknown" / "Uncategorized" when we build the analytical dataset.

### For those 1,603 transactions, let's see how much revenue they represent.

In [26]:
missing_category_sales = order_items[
    order_items["product_id"].isin(missing_cate_products)
]

missing_category_sales["price"].sum()

np.float64(179535.28)

In [27]:
order_items["price"].sum()

np.float64(13591643.7)

## the review text missing values

In [28]:
reviews["review_comment_title"].notna().sum(), reviews["review_comment_message"].notna().sum()

(np.int64(11568), np.int64(40977))

| Field                    | Present | Missing |  % Present |
| ------------------------ | ------: | ------: | ---------: |
| `review_comment_title`   |  11,568 |  87,656 | **11.66%** |
| `review_comment_message` |  40,977 |  58,247 | **41.30%** |

So the majority of customers didn't write a title, and about 41% provided a written message.

### We should leave these NaNs as they are.

Orders

Missing timestamps → retain as NaT; they have operational meaning.

Order items

No missing values → no action.

Products

610 products have incomplete descriptive metadata → retain.
1,603 transactions involve those products → definitely don't delete the transactions.
Missing categories will eventually become "Unknown" in the analytical layer, not necessarily overwrite the raw value.

Payments

No missing values → no action.

Reviews

Missing comments → valid absence of written feedback → retain as NaN.

In [29]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


In [30]:
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [31]:
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"]
)

reviews["review_creation_date"] = pd.to_datetime(
    reviews["review_creation_date"]
)

reviews["review_answer_timestamp"] = pd.to_datetime(
    reviews["review_answer_timestamp"]
)

In [32]:
order_items.info()
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 6.0 MB
<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  str           
 1   order_id         

### Next: validate the numeric columns

Before we start creating derived columns or SQL tables, we should make sure the numeric data actually makes sense.

#### first lets check extreme values

In [33]:
print("Negative prices:", (order_items["price"] < 0).sum())
print("Zero prices:", (order_items["price"] == 0).sum())

print("Negative freight:", (order_items["freight_value"] < 0).sum())
print("Zero freight:", (order_items["freight_value"] == 0).sum())

Negative prices: 0
Zero prices: 0
Negative freight: 0
Zero freight: 383


In [34]:
print("Review scores outside 1-5:",
      ((reviews["review_score"] < 1) |
       (reviews["review_score"] > 5)).sum())

Review scores outside 1-5: 0


In [35]:
print(
    "Negative payment values:",
    (payments["payment_value"] < 0).sum()
)

print(
    "Zero payment values:",
    (payments["payment_value"] == 0).sum()
)

print(
    "Invalid installments:",
    (payments["payment_installments"] <= 0).sum()
)

Negative payment values: 0
Zero payment values: 9
Invalid installments: 2


In [36]:
payments[
    payments["payment_value"] == 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [37]:
payments[
    payments["payment_installments"] <= 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


### we've finally found some genuine anomalies.
A zero payment_value isn't necessarily something we should delete. It could represent a payment record with no monetary amount, particularly around voucher/payment handling.

For a credit-card payment, 0 installments doesn't make business sense under the usual interpretation of this field.

| Issue                  | Count | Decision                                                              |
| ---------------------- | ----: | --------------------------------------------------------------------- |
| Negative payment value |     0 | No issue                                                              |
| Zero payment value     |     9 | **Keep**                                                              |
| Installments ≤ 0       |     2 | **Keep original records; exclude from installment-specific analysis** |



In [38]:
payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

The only unusual value is not_defined

There are only 3 records with it.

For payment-method analysis, we can either:

show them as Not Defined, or
group them under Other in a dashboard if appropriate.



### 📊 Data-Quality Audit: First-Pass Summary

We have completed a comprehensive first-pass data-quality audit across our datasets. Here is the current status of our data cleaning and validation efforts:

#### 🛒 Orders
* **Datetimes converted** ✅
* **Missing dates investigated** ✅
* **Status values validated** ✅

#### 👥 Customers
* **State values validated** ✅
* **City values validated programmatically** ✅
* **No whitespace/casing issues** ✅

#### 📦 Order Items
* **Datetime converted** ✅
* **No missing values** ✅
* **No negative/zero prices** ✅
* **Zero freight investigated and retained** ✅

#### 🏷️ Products
* **Missing metadata investigated** ✅
* **Missing-category products linked back to transactions** ✅
  * *Impact:* 1,603 transactions affected (~1.32% of item sales).
* **Missing physical attributes investigated** ✅

#### 💳 Payments
* **No missing values** ✅
* **Negative values:** None found ✅
* **Zero-value payments:** Investigated ✅
* **Invalid records:** 2 invalid installment records identified ⚠️
* **Payment types validated** ✅

#### ⭐ Reviews
* **Datetimes converted** ✅
* **Review scores validated** Verified against a 1–5 scale ✅
* **Missing comments understood** Confirmed as optional feedback ✅


In [39]:
orders.to_csv("../data/processed/orders_cleaned.csv", index=False)
order_items.to_csv("../data/processed/order_items_cleaned.csv", index=False)
products.to_csv("../data/processed/products_cleaned.csv", index=False)
customers.to_csv("../data/processed/customers_cleaned.csv", index=False)
sellers.to_csv("../data/processed/sellers_cleaned.csv", index=False)
payments.to_csv("../data/processed/payments_cleaned.csv", index=False)
reviews.to_csv("../data/processed/reviews_cleaned.csv", index=False)